# CURE-Rec — behavioral calibration robustness

This notebook runs the **pre-specified CURE-Sim assumption-sensitivity study** after the final BPR archive is frozen. It does not use MovieLens to calibrate causal effects and it does not tune the recommendation model.

Each enabled run recomputes the exact six-player game for every configuration and seed. Leave all execution guards `False` until you deliberately want that cost.


## 1. Setup

Run after `git pull` and a kernel restart. The cell resolves the CURE-Rec source directory from either the repository root or the code directory.


In [1]:
from pathlib import Path
import importlib
import json
import sys
import pandas as pd

CWD = Path.cwd().resolve()
CANDIDATES = [CWD, CWD / 'paper-ideas' / 'CURE-Rec' / 'code', *CWD.parents]
ROOT = next((p for p in CANDIDATES if (p / 'pyproject.toml').exists() and (p / 'cure_rec').exists()), None)
if ROOT is None:
    raise RuntimeError('Open this notebook from the CURE-Rec code directory or repository root.')
sys.path[:] = [str(ROOT), *[item for item in sys.path if item != str(ROOT)]]
for name in list(sys.modules):
    if name == 'cure_rec' or name.startswith('cure_rec.'):
        del sys.modules[name]
importlib.invalidate_caches()

from cure_rec.calibration import run_calibration_sweep
from cure_rec.config import load_settings

QUICK_CONFIG = ROOT / 'configs' / 'curesim_quickstart.yaml'
FULL_CONFIG = ROOT / 'configs' / 'curesim_full.yaml'
RUN_ROOT = ROOT / 'runs'
print('CURE-Rec source:', ROOT)


CURE-Rec source: /Users/mlouhichi/Desktop/CURE-Rec/next-paper/paper-ideas/CURE-Rec/code


## 2. Read this before enabling a run

- **OAT** runs one frozen baseline and two predeclared contrast values for each of seven assumptions: 15 configurations total.
- **LHS** runs one frozen baseline plus the requested number of joint Latin-hypercube configurations.
- Each configuration is repeated over the stated independent environment seeds and evaluates all 64 coalitions in every configured scenario.
- Neither design selects a best configuration. All seed-level results, feasibility outcomes, planner modes, attributions, and interactions are retained.


In [11]:
RUN_CALIBRATION_SMOKE = False
RUN_CALIBRATION_OAT_FULL = False
RUN_CALIBRATION_LHS_FULL = True

CALIBRATION_RESUME_DIR = None


## 3. Action 1 — cheap structural smoke run

This is still an exact-game run, but uses the quick simulator, one seed, and one LHS configuration plus the baseline. It verifies output tables/figures before the full study.


In [8]:
if RUN_CALIBRATION_SMOKE:
    smoke = load_settings(QUICK_CONFIG)
    smoke.run.name = 'calibration-smoke'
    smoke.run.output_root = RUN_ROOT
    smoke_result = run_calibration_sweep(smoke, seeds=(42,), design='lhs', lhs_samples=1)
    print('Smoke calibration:', smoke_result.run_dir)
    display(smoke_result.summary)
else:
    print('Smoke calibration disabled.')


Smoke calibration disabled.


## 4. Action 2 — full one-at-a-time phase diagram

This is the interpretable primary sensitivity analysis. It evaluates 15 configurations × 5 seeds under the full four-scenario CURE-Sim configuration. It checkpoints completed point-level aggregates and can resume after a shutdown. To resume an interrupted run, set `CALIBRATION_RESUME_DIR` in the guard cell; completed child seeds are recovered and are not rerun.


In [7]:
if RUN_CALIBRATION_OAT_FULL:
    full = load_settings(FULL_CONFIG)
    full.run.name = 'calibration-oat-full'
    full.run.output_root = RUN_ROOT
    oat_result = run_calibration_sweep(full, seeds=FULL_SEEDS, design='oat', resume_dir=CALIBRATION_RESUME_DIR)
    print('Full OAT calibration:', oat_result.run_dir)
    display(oat_result.summary)
else:
    print('Full OAT calibration disabled.')


2026-08-06 14:18:57,454 | INFO | run_started | {"config_hash": "8f50e9250a7573aa", "run_id": "calibration-oat-provider_threshold-0p3200-seed-46-20260806T131857Z-c092d17b"}
2026-08-06 14:18:57,454 | INFO | exact_game_started | {}
2026-08-06 14:18:57,455 | INFO | simulator_ready | {"horizon": 12, "n_items": 240, "n_users": 120, "scenario": "nominal"}
2026-08-06 14:21:37,864 | INFO | scenario_game_completed | {"grand_coalition_improvement": -0.13812789327397867, "scenario": "nominal", "shapley_efficiency_gap": 0.0}
2026-08-06 14:21:37,865 | INFO | simulator_ready | {"horizon": 12, "n_items": 240, "n_users": 120, "scenario": "fatigue_stress"}
2026-08-06 14:24:16,499 | INFO | scenario_game_completed | {"grand_coalition_improvement": -0.13526674777644043, "scenario": "fatigue_stress", "shapley_efficiency_gap": 8.326672684688674e-17}
2026-08-06 14:24:16,500 | INFO | simulator_ready | {"horizon": 12, "n_items": 240, "n_users": 120, "scenario": "popularity_stress"}
2026-08-06 14:26:55,135 | INF

,point_id,varied_parameter,varied_value,is_baseline,seed_count,selected_portfolio_mode,selected_portfolio_mode_frequency,selection_stability,repeat_cap_selection_rate,base_feasibility_rate,...,fatigue_upper_mean,config_hash,recovered_seed_count,parameter_fatigue_strength,parameter_repeat_threshold,parameter_horizon,parameter_provider_threshold,parameter_provider_balance_strength,parameter_novelty_delayed_benefit,parameter_exploration_cost
0,baseline,baseline,NaN,True,5,"('repeat_cap',)",5,1.0,1.0,0.2,...,0.000000,9798e91ec7ada5f2,5,1.00,3,12,0.28,0.35,0.00,0.10
1,oat-fatigue_strength-0p7500,fatigue_strength,0.75,False,5,"('repeat_cap',)",5,1.0,1.0,0.2,...,0.000000,aad136c8858c57bf,5,0.75,3,12,0.28,0.35,0.00,0.10
2,oat-fatigue_strength-1p2500,fatigue_strength,1.25,False,5,"('repeat_cap',)",5,1.0,1.0,0.2,...,0.000000,23b22ca44e5e1491,5,1.25,3,12,0.28,0.35,0.00,0.10
3,oat-repeat_threshold-2,repeat_threshold,2.00,False,5,"('repeat_cap',)",5,1.0,1.0,0.2,...,0.262137,0e8c4922cb348567,5,1.00,2,12,0.28,0.35,0.00,0.10
4,oat-repeat_threshold-4,repeat_threshold,4.00,False,5,"('repeat_cap',)",5,1.0,1.0,0.2,...,0.000000,409a3894e128ae19,5,1.00,4,12,0.28,0.35,0.00,0.10
5,oat-horizon-8,horizon,8.00,False,5,"('repeat_cap',)",5,1.0,1.0,0.6,...,0.000000,3dfc6af2b625c550,5,1.00,3,8,0.28,0.35,0.00,0.10
6,oat-horizon-16,horizon,16.00,False,5,"('repeat_cap',)",5,1.0,1.0,0.2,...,0.000000,cdfbd1be96de4893,5,1.00,3,16,0.28,0.35,0.00,0.10
7,oat-provider_threshold-0p2400,provider_threshold,0.24,False,5,"('repeat_cap',)",4,0.8,1.0,0.0,...,0.000000,7796d9c704e67468,5,1.00,3,12,0.24,0.35,0.00,0.10
8,oat-provider_threshold-0p3200,provider_threshold,0.32,False,5,"('repeat_cap',)",5,1.0,1.0,0.6,...,0.000000,7b1e304238e1cae7,5,1.00,3,12,0.32,0.35,0.00,0.10
9,oat-provider_balance_strength-0p1500,provider_balance_strength,0.15,False,5,"('repeat_cap',)",5,1.0,1.0,0.2,...,0.000000,658db72cd92964cf,5,1.00,3,12,0.28,0.15,0.00,0.10


## 5. Action 3 — joint Latin-hypercube robustness probe

Run only after the OAT phase diagram has completed. This examines joint assumptions rather than picking a favorable setting. With `LHS_SAMPLES = 24`, it evaluates 25 configurations (including the baseline) × 5 seeds.


In [12]:
if RUN_CALIBRATION_LHS_FULL:
    full = load_settings(FULL_CONFIG)
    full.run.name = 'calibration-lhs-full'
    full.run.output_root = RUN_ROOT
    lhs_result = run_calibration_sweep(full, seeds=FULL_SEEDS, design='lhs', lhs_samples=LHS_SAMPLES, resume_dir=CALIBRATION_RESUME_DIR)
    print('Full LHS calibration:', lhs_result.run_dir)
    display(lhs_result.summary)
else:
    print('Full LHS calibration disabled.')


2026-08-06 20:54:46,771 | INFO | run_started | {"config_hash": "f069f4f0ed51dcaf", "run_id": "calibration-baseline-seed-42-20260806T195446Z-a7ee83c5"}
2026-08-06 20:54:46,772 | INFO | exact_game_started | {}
2026-08-06 20:54:46,773 | INFO | simulator_ready | {"horizon": 12, "n_items": 240, "n_users": 120, "scenario": "nominal"}
2026-08-06 20:57:29,636 | INFO | scenario_game_completed | {"grand_coalition_improvement": -0.13710546477148466, "scenario": "nominal", "shapley_efficiency_gap": 0.0}
2026-08-06 20:57:29,637 | INFO | simulator_ready | {"horizon": 12, "n_items": 240, "n_users": 120, "scenario": "fatigue_stress"}
2026-08-06 21:00:11,939 | INFO | scenario_game_completed | {"grand_coalition_improvement": -0.13194331906701645, "scenario": "fatigue_stress", "shapley_efficiency_gap": 2.7755575615628914e-17}
2026-08-06 21:00:11,940 | INFO | simulator_ready | {"horizon": 12, "n_items": 240, "n_users": 120, "scenario": "popularity_stress"}
2026-08-06 21:02:54,353 | INFO | scenario_game_co

,point_id,varied_parameter,varied_value,is_baseline,seed_count,selected_portfolio_mode,selected_portfolio_mode_frequency,selection_stability,repeat_cap_selection_rate,base_feasibility_rate,...,fatigue_upper_mean,config_hash,recovered_seed_count,parameter_fatigue_strength,parameter_repeat_threshold,parameter_horizon,parameter_provider_threshold,parameter_provider_balance_strength,parameter_novelty_delayed_benefit,parameter_exploration_cost
0,baseline,baseline,None,True,5,"('repeat_cap',)",5,1.0,1.0,0.2,...,0.000000,1aada97a740787be,5,1.000000,3,12,0.280000,0.350000,0.000000,0.100000
1,lhs-001,joint_lhs,None,False,5,"('repeat_cap', 'tail_slot')",5,1.0,1.0,0.0,...,0.637762,069297af278b50e1,5,1.076705,1,17,0.324179,0.468345,0.060859,0.160875
2,lhs-002,joint_lhs,None,False,5,"('repeat_cap',)",4,0.8,1.0,0.2,...,0.226724,93eccbe1c389e91d,5,1.367944,2,10,0.262036,0.386883,0.070208,0.064656
3,lhs-003,joint_lhs,None,False,5,"('repeat_cap', 'explore_slot')",4,0.8,1.0,0.2,...,0.455595,768e4245a7319c24,5,1.047772,1,10,0.252429,0.327820,0.058298,0.025139
4,lhs-004,joint_lhs,None,False,5,"('repeat_cap',)",5,1.0,1.0,0.0,...,0.000000,ac4261f4e750d1db,5,0.761171,5,17,0.277819,0.423906,0.011495,0.079573
5,lhs-005,joint_lhs,None,False,5,"('repeat_cap',)",4,0.8,1.0,0.2,...,0.000000,d7fa97922454eaf5,5,1.129346,3,9,0.267317,0.297299,0.031857,0.067052
6,lhs-006,joint_lhs,None,False,5,"('repeat_cap',)",5,1.0,1.0,1.0,...,0.244154,bc4f57cc3729dcd6,5,0.870287,2,11,0.340421,0.795592,0.005958,0.037446
7,lhs-007,joint_lhs,None,False,5,"('repeat_cap',)",5,1.0,1.0,1.0,...,0.000000,4c0afd79d7336391,5,1.242046,5,8,0.348143,0.737506,0.007593,0.100119
8,lhs-008,joint_lhs,None,False,5,(),3,0.6,0.4,0.6,...,0.050975,69fd374f5f7be583,5,1.184850,4,6,0.257343,0.020907,0.077012,0.033885
9,lhs-009,joint_lhs,None,False,5,"('repeat_cap', 'tail_slot')",2,0.4,1.0,0.0,...,0.000000,b9dedc54492fc8e2,5,1.094094,3,11,0.213379,0.348619,0.018867,0.188931


## 6. Action 4 — inspect a completed calibration run

Paste the run directory printed by Action 2 or 3. This operation is cheap and never re-runs coalitions.


In [9]:
CALIBRATION_OUTPUT = (
    RUN_ROOT / "calibration-oat-20260805T184133Z"
)
if CALIBRATION_OUTPUT is not None:
    CALIBRATION_OUTPUT = Path(CALIBRATION_OUTPUT)
    calibration_summary = pd.read_csv(CALIBRATION_OUTPUT / 'calibration_summary.csv')
    calibration_configurations = pd.read_csv(CALIBRATION_OUTPUT / 'calibration_configurations.csv')
    calibration_manifest = json.loads((CALIBRATION_OUTPUT / 'calibration_manifest.json').read_text())
    display(calibration_summary)
    display(calibration_configurations)
    print(json.dumps({
        'design': calibration_manifest['design'],
        'seeds': calibration_manifest['seeds'],
        'base_config_hash': calibration_manifest['base_config_hash'],
        'figures': sorted(path.name for path in (CALIBRATION_OUTPUT / 'figures').glob('*.png')),
    }, indent=2))
else:
    print('Set CALIBRATION_OUTPUT after a calibration run to inspect its artifacts.')


,point_id,varied_parameter,varied_value,is_baseline,seed_count,selected_portfolio_mode,selected_portfolio_mode_frequency,selection_stability,repeat_cap_selection_rate,base_feasibility_rate,...,fatigue_upper_mean,config_hash,recovered_seed_count,parameter_fatigue_strength,parameter_repeat_threshold,parameter_horizon,parameter_provider_threshold,parameter_provider_balance_strength,parameter_novelty_delayed_benefit,parameter_exploration_cost
0,baseline,baseline,NaN,True,5,"('repeat_cap',)",5,1.0,1.0,0.2,...,0.000000,9798e91ec7ada5f2,5,1.00,3,12,0.28,0.35,0.00,0.10
1,oat-fatigue_strength-0p7500,fatigue_strength,0.75,False,5,"('repeat_cap',)",5,1.0,1.0,0.2,...,0.000000,aad136c8858c57bf,5,0.75,3,12,0.28,0.35,0.00,0.10
2,oat-fatigue_strength-1p2500,fatigue_strength,1.25,False,5,"('repeat_cap',)",5,1.0,1.0,0.2,...,0.000000,23b22ca44e5e1491,5,1.25,3,12,0.28,0.35,0.00,0.10
3,oat-repeat_threshold-2,repeat_threshold,2.00,False,5,"('repeat_cap',)",5,1.0,1.0,0.2,...,0.262137,0e8c4922cb348567,5,1.00,2,12,0.28,0.35,0.00,0.10
4,oat-repeat_threshold-4,repeat_threshold,4.00,False,5,"('repeat_cap',)",5,1.0,1.0,0.2,...,0.000000,409a3894e128ae19,5,1.00,4,12,0.28,0.35,0.00,0.10
5,oat-horizon-8,horizon,8.00,False,5,"('repeat_cap',)",5,1.0,1.0,0.6,...,0.000000,3dfc6af2b625c550,5,1.00,3,8,0.28,0.35,0.00,0.10
6,oat-horizon-16,horizon,16.00,False,5,"('repeat_cap',)",5,1.0,1.0,0.2,...,0.000000,cdfbd1be96de4893,5,1.00,3,16,0.28,0.35,0.00,0.10
7,oat-provider_threshold-0p2400,provider_threshold,0.24,False,5,"('repeat_cap',)",4,0.8,1.0,0.0,...,0.000000,7796d9c704e67468,5,1.00,3,12,0.24,0.35,0.00,0.10
8,oat-provider_threshold-0p3200,provider_threshold,0.32,False,5,"('repeat_cap',)",5,1.0,1.0,0.6,...,0.000000,7b1e304238e1cae7,5,1.00,3,12,0.32,0.35,0.00,0.10
9,oat-provider_balance_strength-0p1500,provider_balance_strength,0.15,False,5,"('repeat_cap',)",5,1.0,1.0,0.2,...,0.000000,658db72cd92964cf,5,1.00,3,12,0.28,0.15,0.00,0.10


,point_id,varied_parameter,varied_value,is_baseline,config_hash,sweep_run_dir,recovered_seed_count,parameter_fatigue_strength,parameter_repeat_threshold,parameter_horizon,parameter_provider_threshold,parameter_provider_balance_strength,parameter_novelty_delayed_benefit,parameter_exploration_cost
0,baseline,baseline,NaN,True,9798e91ec7ada5f2,/Users/mlouhichi/Desktop/CURE-Rec/next-paper/p...,5,1.00,3,12,0.28,0.35,0.00,0.10
1,oat-fatigue_strength-0p7500,fatigue_strength,0.75,False,aad136c8858c57bf,/Users/mlouhichi/Desktop/CURE-Rec/next-paper/p...,5,0.75,3,12,0.28,0.35,0.00,0.10
2,oat-fatigue_strength-1p2500,fatigue_strength,1.25,False,23b22ca44e5e1491,/Users/mlouhichi/Desktop/CURE-Rec/next-paper/p...,5,1.25,3,12,0.28,0.35,0.00,0.10
3,oat-repeat_threshold-2,repeat_threshold,2.00,False,0e8c4922cb348567,/Users/mlouhichi/Desktop/CURE-Rec/next-paper/p...,5,1.00,2,12,0.28,0.35,0.00,0.10
4,oat-repeat_threshold-4,repeat_threshold,4.00,False,409a3894e128ae19,/Users/mlouhichi/Desktop/CURE-Rec/next-paper/p...,5,1.00,4,12,0.28,0.35,0.00,0.10
5,oat-horizon-8,horizon,8.00,False,3dfc6af2b625c550,/Users/mlouhichi/Desktop/CURE-Rec/next-paper/p...,5,1.00,3,8,0.28,0.35,0.00,0.10
6,oat-horizon-16,horizon,16.00,False,cdfbd1be96de4893,/Users/mlouhichi/Desktop/CURE-Rec/next-paper/p...,5,1.00,3,16,0.28,0.35,0.00,0.10
7,oat-provider_threshold-0p2400,provider_threshold,0.24,False,7796d9c704e67468,/Users/mlouhichi/Desktop/CURE-Rec/next-paper/p...,5,1.00,3,12,0.24,0.35,0.00,0.10
8,oat-provider_threshold-0p3200,provider_threshold,0.32,False,7b1e304238e1cae7,/Users/mlouhichi/Desktop/CURE-Rec/next-paper/p...,5,1.00,3,12,0.32,0.35,0.00,0.10
9,oat-provider_balance_strength-0p1500,provider_balance_strength,0.15,False,658db72cd92964cf,/Users/mlouhichi/Desktop/CURE-Rec/next-paper/p...,5,1.00,3,12,0.28,0.15,0.00,0.10


{
  "design": "oat",
  "seeds": [
    42,
    43,
    44,
    45,
    46
  ],
  "base_config_hash": "41b09c43d0bc6c0e",
  "figures": [
    "calibration_figure_decision_stability.png",
    "calibration_figure_oat_lower_improvement.png",
    "calibration_figure_repeat_cap_shapley.png"
  ]
}


## Interpretation gate

Report this study as sensitivity of the disclosed CURE-Sim behavioral model. A stable `repeat_cap` selection rate across these settings supports simulator robustness; it does **not** turn MovieLens ratings into logged causal policy evidence. Retain configurations with unfavorable results, feasibility failures, and repair decisions.
